In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt                                                                     ### PLOTTING
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.spatial import ConvexHull                                                                ### PLOTTING

AR_CODE_DIR = next(
    (
        candidate.resolve()
        for root in (Path.cwd(), *Path.cwd().parents)
        for candidate in (
            root / "Code" / "Code_AR_processes",
            root,
        )
        if (candidate / "functions.py").is_file()
        and (candidate / "plotting_functions.py").is_file()
    ),
    None,
)
if AR_CODE_DIR is None:
    raise FileNotFoundError(
        "Could not locate Code/Code_AR_processes from the current directory."
    )
PROJECT_ROOT = AR_CODE_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

AR_IMAGES_DIR = AR_CODE_DIR / "outputs" / "images"                                                  ### PLOTTING
EXPERIMENT_OVERVIEWS_DIR = AR_IMAGES_DIR / "experiment_overviews"                                   ### PLOTTING
ORDER_SELECTION_IMAGES_DIR = AR_IMAGES_DIR / "order_selection"                                      ### PLOTTING
AR_IMAGE_DIRECTORIES = [                                                                            ### PLOTTING
    EXPERIMENT_OVERVIEWS_DIR,                                                                       ### PLOTTING
    *(ORDER_SELECTION_IMAGES_DIR / name for name in ("ar2", "cross_process")),                      ### PLOTTING
]                                                                                                   ### PLOTTING
for image_directory in AR_IMAGE_DIRECTORIES:                                                        ### PLOTTING
    image_directory.mkdir(parents=True, exist_ok=True)                                              ### PLOTTING

plt.style.use(style="seaborn-v0_8-whitegrid")                                                       ### PLOTTING
np.set_printoptions(precision=4, suppress=True)                                                     ### REPORTING


In [ ]:
import importlib

from Code.Code_AR_processes import functions as ar_functions

ar_functions = importlib.reload(ar_functions)

from Code.Code_AR_processes.functions import (
    jones_ar_coefficients,
    simulate_uniform_ar,
    sliding_window_point_cloud,
    ar_impulse_response,
    projected_subgradient_empirical_width,
    batched_geometric_width_curves,
    theoretical_ar_width_in_direction,
    population_minimizing_ar2_direction,
    two_affine_l_method_selector,
    affine_constant_l_method_selector,
    theoretical_width_2d_from_angle,
    ar_window_zonotope_generators,
    zonotope_polygon_2d,
    fibonacci_sphere_directions,
    zonotope_support_points,
)

In [ ]:
p = 2
coefficient_seed = 42
process_seed = 20260806
sample_size = 1000
burnin = 500
innovations_half_width = 1.0

phi = jones_ar_coefficients(p, rng=coefficient_seed, return_pacf=False)
ar_series = simulate_uniform_ar(phi, sample_size, burnin, innovations_half_width, rng=process_seed)

print("phi:", phi)                                                                                  ### REPORTING

fig, ax = plt.subplots(figsize=(10, 4))                                                             ### PLOTTING
ax.plot(ar_series, color="blue", linewidth=1.5)                                                     ### PLOTTING
ax.set_title(f"Simulated AR({p}) Process", fontsize=14)                                             ### PLOTTING
ax.set_xlabel("Time", fontsize=12)                                                                  ### PLOTTING
ax.set_ylabel("Value", fontsize=12)                                                                 ### PLOTTING
plt.tight_layout()                                                                                  ### PLOTTING
plt.show()                                                                                          ### PLOTTING

In [ ]:
cloud_2d = sliding_window_point_cloud(ar_series, embedding_dimension=2)
cloud_3d = sliding_window_point_cloud(ar_series, embedding_dimension=3)

subgradient_2d = projected_subgradient_empirical_width(
    cloud_2d,
    iterations=2_000,
    initial_step=0.2,
    rng=20260806,
)
subgradient_3d = projected_subgradient_empirical_width(
    cloud_3d,
    iterations=2_000,
    initial_step=0.2,
    rng=20260807,
)

theoretical_history = 10_000
psi_for_cloud_plot = ar_impulse_response(phi, theoretical_history)
psi_shifted_once = np.concatenate(([0.0], psi_for_cloud_plot[:-1]))

angle_grid = np.linspace(0.0, np.pi, 721, endpoint=False)
width_grid = np.array([
    theoretical_width_2d_from_angle(
        angle, psi_for_cloud_plot, psi_shifted_once,
        innovations_half_width,
    )
    for angle in angle_grid
])
best_grid_angle = angle_grid[np.argmin(width_grid)]
theoretical_fit_2d = minimize(
    lambda angle: theoretical_width_2d_from_angle(
        float(angle[0]), psi_for_cloud_plot, psi_shifted_once,
        innovations_half_width,
    ),
    x0=np.array([best_grid_angle]),
    method="Nelder-Mead",
    options={"xatol": 1e-12, "fatol": 1e-12},
)
theoretical_angle_2d = float(theoretical_fit_2d.x[0] % np.pi)
theoretical_direction_2d = np.array([
    np.cos(theoretical_angle_2d),
    np.sin(theoretical_angle_2d),
])
theoretical_minimum_width_2d = theoretical_width_2d_from_angle(
    theoretical_angle_2d, psi_for_cloud_plot, psi_shifted_once,
    innovations_half_width,
)

theoretical_direction_3d = np.concatenate(([1.0], -phi))
theoretical_direction_3d /= np.linalg.norm(theoretical_direction_3d)
generator_projections_3d = np.convolve(
    psi_for_cloud_plot, theoretical_direction_3d
)[: theoretical_history + 1]
theoretical_minimum_width_3d = float(
    2.0 * innovations_half_width * np.sum(np.abs(generator_projections_3d))
)

# Use a finite impulse-response history to obtain drawable approximations                           ### PLOTTING
# Z_(k,H) of the infinite theoretical support zonotopes.                                            ### PLOTTING
zonotope_history = 300                                                                              ### PLOTTING

generators_2d = ar_window_zonotope_generators(                                                      ### PLOTTING
    phi, 2, zonotope_history, innovations_half_width                                                ### PLOTTING
)                                                                                                   ### PLOTTING
generators_3d = ar_window_zonotope_generators(                                                      ### PLOTTING
    phi, 3, zonotope_history, innovations_half_width                                                ### PLOTTING
)                                                                                                   ### PLOTTING
zonotope_vertices_2d = zonotope_polygon_2d(generators_2d)                                           ### PLOTTING

sphere_directions = fibonacci_sphere_directions(1_200)                                              ### PLOTTING
sphere_directions = np.vstack((                                                                     ### PLOTTING
    sphere_directions,                                                                              ### PLOTTING
    theoretical_direction_3d,                                                                       ### PLOTTING
    -theoretical_direction_3d,                                                                      ### PLOTTING
))                                                                                                  ### PLOTTING
zonotope_surface_points_3d = zonotope_support_points(                                               ### PLOTTING
    generators_3d, sphere_directions                                                                ### PLOTTING
)                                                                                                   ### PLOTTING
zonotope_hull_3d = ConvexHull(zonotope_surface_points_3d)                                           ### PLOTTING

# The two support levels are +/-h, so their separation is the finite-history width 2h.              ### PLOTTING
support_half_width_2d = np.sum(                                                                     ### PLOTTING
    np.abs(generators_2d.T @ theoretical_direction_2d)                                              ### PLOTTING
)                                                                                                   ### PLOTTING
support_half_width_3d = np.sum(                                                                     ### PLOTTING
    np.abs(generators_3d.T @ theoretical_direction_3d)                                              ### PLOTTING
)                                                                                                   ### PLOTTING
approximating_width_2d = 2.0 * support_half_width_2d                                                ### PLOTTING
approximating_width_3d = 2.0 * support_half_width_3d                                                ### PLOTTING

perpendicular_2d = np.array([                                                                       ### PLOTTING
    -theoretical_direction_2d[1], theoretical_direction_2d[0]                                       ### PLOTTING
])                                                                                                  ### PLOTTING
perpendicular_coordinates_2d = zonotope_vertices_2d @ perpendicular_2d                              ### PLOTTING
support_line_coordinate = np.linspace(                                                              ### PLOTTING
    perpendicular_coordinates_2d.min(),                                                             ### PLOTTING
    perpendicular_coordinates_2d.max(),                                                             ### PLOTTING
    2,                                                                                              ### PLOTTING
)                                                                                                   ### PLOTTING

reference_axis = np.array([0.0, 0.0, 1.0])                                                          ### PLOTTING
if abs(theoretical_direction_3d @ reference_axis) > 0.9:                                            ### PLOTTING
    reference_axis = np.array([0.0, 1.0, 0.0])                                                      ### PLOTTING
plane_basis_1 = np.cross(theoretical_direction_3d, reference_axis)                                  ### PLOTTING
plane_basis_1 /= np.linalg.norm(plane_basis_1)                                                      ### PLOTTING
plane_basis_2 = np.cross(theoretical_direction_3d, plane_basis_1)                                   ### PLOTTING
plane_coordinate_1 = zonotope_surface_points_3d @ plane_basis_1                                     ### PLOTTING
plane_coordinate_2 = zonotope_surface_points_3d @ plane_basis_2                                     ### PLOTTING
plane_grid_1, plane_grid_2 = np.meshgrid(                                                           ### PLOTTING
    np.linspace(plane_coordinate_1.min(), plane_coordinate_1.max(), 2),                             ### PLOTTING
    np.linspace(plane_coordinate_2.min(), plane_coordinate_2.max(), 2),                             ### PLOTTING
)                                                                                                   ### PLOTTING

direction_half_length_2d = 0.5 * np.linalg.norm(np.ptp(zonotope_vertices_2d, axis=0))               ### PLOTTING
direction_endpoints_2d = np.outer(                                                                  ### PLOTTING
    [-direction_half_length_2d, direction_half_length_2d],                                          ### PLOTTING
    theoretical_direction_2d,                                                                       ### PLOTTING
)                                                                                                   ### PLOTTING
direction_half_length_3d = 0.5 * np.linalg.norm(                                                    ### PLOTTING
    np.ptp(zonotope_surface_points_3d, axis=0)                                                      ### PLOTTING
)                                                                                                   ### PLOTTING
direction_endpoints_3d = np.outer(                                                                  ### PLOTTING
    [-direction_half_length_3d, direction_half_length_3d],                                          ### PLOTTING
    theoretical_direction_3d,                                                                       ### PLOTTING
)                                                                                                   ### PLOTTING

fig = plt.figure(figsize=(14, 6), constrained_layout=True)                                          ### PLOTTING
ax_2d = fig.add_subplot(1, 2, 1)                                                                    ### PLOTTING
ax_3d = fig.add_subplot(1, 2, 2, projection='3d')                                                   ### PLOTTING
ax_2d.fill(                                                                                         ### PLOTTING
    zonotope_vertices_2d[:, 0],                                                                     ### PLOTTING
    zonotope_vertices_2d[:, 1],                                                                     ### PLOTTING
    color="tab:orange",                                                                             ### PLOTTING
    alpha=0.18,                                                                                     ### PLOTTING
    label=fr"approximating zonotope $\mathcal{{Z}}_{{2,{zonotope_history}}}$",                      ### PLOTTING
)                                                                                                   ### PLOTTING
ax_2d.plot(                                                                                         ### PLOTTING
    zonotope_vertices_2d[:, 0],                                                                     ### PLOTTING
    zonotope_vertices_2d[:, 1],                                                                     ### PLOTTING
    color="tab:orange",                                                                             ### PLOTTING
    linewidth=1.5,                                                                                  ### PLOTTING
)                                                                                                   ### PLOTTING
ax_2d.scatter(                                                                                      ### PLOTTING
    cloud_2d[:, 0], cloud_2d[:, 1],                                                                 ### PLOTTING
    color="tab:blue", s=10, alpha=0.6, label=r"point cloud $\mathcal{P}_2$"                         ### PLOTTING
)                                                                                                   ### PLOTTING
for support_sign in (-1.0, 1.0):                                                                    ### PLOTTING
    support_line = (                                                                                ### PLOTTING
        support_sign * support_half_width_2d * theoretical_direction_2d                             ### PLOTTING
        + np.outer(support_line_coordinate, perpendicular_2d)                                       ### PLOTTING
    )                                                                                               ### PLOTTING
    ax_2d.plot(                                                                                     ### PLOTTING
        support_line[:, 0], support_line[:, 1],                                                     ### PLOTTING
        color="tab:red", linestyle="-.", linewidth=2.0,                                             ### PLOTTING
        label="supporting lines" if support_sign < 0 else None,                                     ### PLOTTING
    )                                                                                               ### PLOTTING
ax_2d.plot(                                                                                         ### PLOTTING
    direction_endpoints_2d[:, 0],                                                                   ### PLOTTING
    direction_endpoints_2d[:, 1],                                                                   ### PLOTTING
    color="black",                                                                                  ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    linewidth=3,                                                                                    ### PLOTTING
    label=r"minimizing direction $u_2^\star$",                                                      ### PLOTTING
)                                                                                                   ### PLOTTING
ax_2d.set_title(                                                                                    ### PLOTTING
    rf"$\mathcal{{P}}_2$ and $\mathcal{{Z}}_{{2,{zonotope_history}}}$: "                            ### PLOTTING
    rf"$w_{{2,{zonotope_history}}}(u_2^\star)={approximating_width_2d:.3f}$",                       ### PLOTTING
    fontsize=12,                                                                                    ### PLOTTING
)                                                                                                   ### PLOTTING
ax_2d.set_xlabel("X (t)", fontsize=10)                                                              ### PLOTTING
ax_2d.set_ylabel("Y (t-1)", fontsize=10)                                                            ### PLOTTING
ax_2d.set_aspect("equal", adjustable="box")                                                         ### PLOTTING
ax_2d.legend(loc="upper right")                                                                     ### PLOTTING
ax_3d.plot_trisurf(                                                                                 ### PLOTTING
    zonotope_surface_points_3d[:, 0],                                                               ### PLOTTING
    zonotope_surface_points_3d[:, 1],                                                               ### PLOTTING
    zonotope_surface_points_3d[:, 2],                                                               ### PLOTTING
    triangles=zonotope_hull_3d.simplices,                                                           ### PLOTTING
    color="tab:orange", alpha=0.12, linewidth=0.15, shade=False,                                    ### PLOTTING
)                                                                                                   ### PLOTTING
ax_3d.scatter([], [], [], color="tab:orange", marker="s", alpha=0.35,                               ### PLOTTING
              label=fr"approximating zonotope $\mathcal{{Z}}_{{3,{zonotope_history}}}$")            ### PLOTTING
ax_3d.scatter(                                                                                      ### PLOTTING
    cloud_3d[:, 0], cloud_3d[:, 1], cloud_3d[:, 2],                                                 ### PLOTTING
    color="tab:green", s=10, alpha=0.6, label=r"point cloud $\mathcal{P}_3$"                        ### PLOTTING
)                                                                                                   ### PLOTTING
for support_sign in (-1.0, 1.0):                                                                    ### PLOTTING
    support_plane = (                                                                               ### PLOTTING
        support_sign * support_half_width_3d * theoretical_direction_3d                             ### PLOTTING
        + plane_grid_1[..., None] * plane_basis_1                                                   ### PLOTTING
        + plane_grid_2[..., None] * plane_basis_2                                                   ### PLOTTING
    )                                                                                               ### PLOTTING
    ax_3d.plot_surface(                                                                             ### PLOTTING
        support_plane[:, :, 0],                                                                     ### PLOTTING
        support_plane[:, :, 1],                                                                     ### PLOTTING
        support_plane[:, :, 2],                                                                     ### PLOTTING
        color="tab:red", alpha=0.12, edgecolor="tab:red", linewidth=0.8,                            ### PLOTTING
    )                                                                                               ### PLOTTING
ax_3d.scatter([], [], [], color="tab:red", marker="s", alpha=0.35,                                  ### PLOTTING
              label="supporting planes")                                                            ### PLOTTING
ax_3d.plot(                                                                                         ### PLOTTING
    direction_endpoints_3d[:, 0],                                                                   ### PLOTTING
    direction_endpoints_3d[:, 1],                                                                   ### PLOTTING
    direction_endpoints_3d[:, 2],                                                                   ### PLOTTING
    color="black",                                                                                  ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    linewidth=3,                                                                                    ### PLOTTING
    label=r"minimizing direction $u_3^\star$",                                                      ### PLOTTING
)                                                                                                   ### PLOTTING
ax_3d.set_title(                                                                                    ### PLOTTING
    rf"$\mathcal{{P}}_3$ and $\mathcal{{Z}}_{{3,{zonotope_history}}}$: "                            ### PLOTTING
    rf"$w_{{3,{zonotope_history}}}(u_3^\star)={approximating_width_3d:.3f}$",                       ### PLOTTING
    fontsize=12,                                                                                    ### PLOTTING
)                                                                                                   ### PLOTTING
ax_3d.set_xlabel("X (t)", fontsize=10)                                                              ### PLOTTING
ax_3d.set_ylabel("Y (t-1)", fontsize=10)                                                            ### PLOTTING
ax_3d.set_zlabel("Z (t-2)", fontsize=10)                                                            ### PLOTTING
ax_3d.set_box_aspect(np.ptp(zonotope_surface_points_3d, axis=0))                                    ### PLOTTING
ax_3d.legend(loc="upper right")                                                                     ### PLOTTING
plt.show()                                                                                          ### PLOTTING

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)                          ### PLOTTING

for axis, result, dimension in zip(                                                                 ### PLOTTING
    axes,                                                                                           ### PLOTTING
    (subgradient_2d, subgradient_3d),                                                               ### PLOTTING
    (2, 3),                                                                                         ### PLOTTING
):                                                                                                  ### PLOTTING
    iteration_numbers = np.arange(1, len(result["width_history"]) + 1)                              ### PLOTTING
    axis.plot(                                                                                      ### PLOTTING
        iteration_numbers,                                                                          ### PLOTTING
        result["width_history"],                                                                    ### PLOTTING
        color="tab:blue",                                                                           ### PLOTTING
        alpha=0.3,                                                                                  ### PLOTTING
        linewidth=0.8,                                                                              ### PLOTTING
        label="current width",                                                                      ### PLOTTING
    )                                                                                               ### PLOTTING
    axis.plot(                                                                                      ### PLOTTING
        iteration_numbers,                                                                          ### PLOTTING
        result["best_width_history"],                                                               ### PLOTTING
        color="tab:red",                                                                            ### PLOTTING
        linewidth=2.0,                                                                              ### PLOTTING
        label="best width so far",                                                                  ### PLOTTING
    )                                                                                               ### PLOTTING
    axis.set(                                                                                       ### PLOTTING
        title=f"Projected subgradient descent in {dimension}D",                                     ### PLOTTING
        xlabel="iteration",                                                                         ### PLOTTING
        ylabel="empirical width",                                                                   ### PLOTTING
    )                                                                                               ### PLOTTING
    axis.legend()                                                                                   ### PLOTTING

plt.show()                                                                                          ### PLOTTING

print("2D minimum width:", subgradient_2d["minimum_width"])                                         ### REPORTING
print("2D minimizing direction:", subgradient_2d["direction"])                                      ### REPORTING
print("3D minimum width:", subgradient_3d["minimum_width"])                                         ### REPORTING
print("3D minimizing direction:", subgradient_3d["direction"])                                      ### REPORTING


## Step-size schedule comparison

The following paired experiment compares the two schedules on the same simulated AR(2) series and the same initial directions. It uses five starts for each embedding dimension, 1,000 iterations, $\eta_0=0.2$, and $\beta=0.99$. Thus the only difference within each pair is

$$\eta_t=\frac{\eta_0}{\sqrt{t+1}} \qquad\text{or}\qquad \eta_t=\eta_0\beta^t.$$


In [ ]:
from time import perf_counter
from IPython.display import display                                                                 ### REPORTING

schedule_comparison_dimensions = [2, 3, 5, 10, 25, 100]
schedule_comparison_starts = 5
schedule_comparison_iterations = 1_000
schedule_comparison_initial_step = 0.2
schedule_comparison_geometric_decay = 0.99
schedule_comparison_rng = np.random.default_rng(20260813)
schedule_comparison_rows = []

for dimension in schedule_comparison_dimensions:
    cloud = sliding_window_point_cloud(ar_series, dimension)
    initial_directions = schedule_comparison_rng.normal(
        size=(schedule_comparison_starts, dimension)
    )
    initial_directions /= np.linalg.norm(
        initial_directions, axis=1, keepdims=True
    )

    for start_index, initial_direction in enumerate(initial_directions):
        for step_schedule in ("inverse_sqrt", "geometric"):
            start_time = perf_counter()
            fit = projected_subgradient_empirical_width(
                cloud,
                iterations=schedule_comparison_iterations,
                initial_step=schedule_comparison_initial_step,
                step_schedule=step_schedule,
                geometric_decay=schedule_comparison_geometric_decay,
                patience=schedule_comparison_iterations,
                improvement_tolerance=0.0,
                initial_direction=initial_direction,
                rng=1_000 + dimension + start_index,
            )
            schedule_comparison_rows.append({
                "k": dimension,
                "start": start_index,
                "step_schedule": step_schedule,
                "minimum_width": fit["minimum_width"],
                "seconds": perf_counter() - start_time,
            })

schedule_comparison = pd.DataFrame(schedule_comparison_rows)
paired_widths = schedule_comparison.pivot(
    index=["k", "start"],
    columns="step_schedule",
    values="minimum_width",
)
median_widths = paired_widths.groupby(level="k").median()
schedule_comparison_summary = median_widths.copy()
schedule_comparison_summary["geometric_wins"] = (
    paired_widths["geometric"] < paired_widths["inverse_sqrt"]
).groupby(level="k").sum()
schedule_comparison_summary["relative_improvement_percent"] = (
    100.0
    * (schedule_comparison_summary["inverse_sqrt"]
       - schedule_comparison_summary["geometric"])
    / schedule_comparison_summary["inverse_sqrt"]
)

display(schedule_comparison_summary.round(6))                                                       ### REPORTING
print(                                                                                              ### REPORTING
    "Overall paired wins for the geometric schedule:",                                              ### REPORTING
    int((paired_widths["geometric"] < paired_widths["inverse_sqrt"]).sum()),                        ### REPORTING
    "out of",                                                                                       ### REPORTING
    len(paired_widths),                                                                             ### REPORTING
)                                                                                                   ### REPORTING


With seed 20260813, the geometric schedule won all 30 paired trials. The observed median minimum widths were:

| $k$ | Geometric | Inverse square root | Geometric wins |
|---:|---:|---:|---:|
| 2 | 8.803711 | 8.803923 | 5/5 |
| 3 | 1.136173 | 1.157649 | 5/5 |
| 5 | 1.113758 | 1.328418 | 5/5 |
| 10 | 1.153020 | 1.509784 | 5/5 |
| 25 | 1.111650 | 1.809104 | 5/5 |
| 100 | 1.250909 | 2.770756 | 5/5 |

For this fixed iteration budget and tuning, the geometric schedule is clearly better. Its faster decay allows fine local adjustments near a kink, whereas the inverse-square-root step is still approximately $0.0063$ at iteration 1,000 and continues to jump across the nonsmooth minimum. This is an empirical comparison, not a universal guarantee: Theorem 4 is local and its admissible values of $\eta_0$ and $\beta$ depend on the sharpness and initialization constants.


## Unpenalized elbow fits for $p=1,2,3$: 50 processes per order

This first methodological comparison retains exactly two selectors and gives neither an order nor a dimension penalty. Both are fitted directly to the empirical minimum-width curve.

1. **Classical two-affine-line L-method.** For every admissible boundary, fit one affine line to each of two disjoint segments, require at least two points per segment, and minimize the segment-size-weighted RMSE. If the boundary is between $k=p$ and $k=p+1$, the selected order is $\widehat p=p$.

2. **Modified affine-plus-constant L-method.** For every candidate plateau onset $q$, fit an affine head through $k\le q$ and a constant tail through $k\ge q$. The shared point $q$ makes $q=2$ admissible, so this method can select AR(1). It uses the same segment-size-weighted RMSE principle.

The experiment below uses the same simulated curves for both fits: 50 processes for each true order, $T=2{,}000$, and embedding dimensions $k=1,\ldots,10$.


In [ ]:
elbow_settings = [
    {"true_order": 1, "seed": 20260813},
    {"true_order": 2, "seed": 20260814},
    {"true_order": 3, "seed": 20260816},
]
elbow_rows = []
elbow_sample_size_T = 2_000
elbow_k_values = np.arange(1, 11)
for setting in elbow_settings:
    true_order = setting["true_order"]
    experiment_seed = setting["seed"]
    for replication in range(50):
        process_generator = np.random.default_rng(np.random.SeedSequence([
            experiment_seed, replication, 0
        ]))
        phi_replication = jones_ar_coefficients(
            true_order, rng=process_generator
        )
        series_replication = simulate_uniform_ar(
            phi_replication,
            sample_size=elbow_sample_size_T,
            burnin=1_000,
            half_width=1.0,
            rng=process_generator,
        )
        _, minimized_widths, _ = batched_geometric_width_curves(
            series_replication,
            elbow_k_values,
            seed=experiment_seed,
            replication=replication,
            starts=3,
            iterations=500,
            initial_step=0.2,
            geometric_decay=0.99,
        )
        selections = {
            "Two affine lines": two_affine_l_method_selector(
                elbow_k_values, minimized_widths
            ),
            "Affine + constant": affine_constant_l_method_selector(
                elbow_k_values, minimized_widths
            ),
        }
        for method, selection in selections.items():
            elbow_rows.append({
                "true_order": true_order,
                "replication": replication + 1,
                "method": method,
                "selected_q": selection["selected_q"],
                "selected_order": selection["selected_order"],
                "fit_score": selection["score"],
                "correct_order": selection["selected_order"] == true_order,
            })
        if (replication + 1) % 10 == 0:                                                             ### REPORTING
            print(                                                                                  ### REPORTING
                f"Completed {replication + 1}/50 AR({true_order}) processes"                        ### REPORTING
            )                                                                                       ### REPORTING

elbow_results = pd.DataFrame(elbow_rows)
method_order = ["Two affine lines", "Affine + constant"]
elbow_accuracy = (
    elbow_results.groupby(["method", "true_order"])["correct_order"]
    .sum().unstack(fill_value=0).reindex(method_order)
)

fig, axis = plt.subplots(figsize=(9.5, 5.2), constrained_layout=True)                              ### PLOTTING
x_positions = np.arange(3)                                                                        ### PLOTTING
bar_width = 0.36                                                                                  ### PLOTTING
for method_index, method in enumerate(method_order):                                              ### PLOTTING
    counts = elbow_accuracy.loc[method, [1, 2, 3]].to_numpy()                                     ### PLOTTING
    offsets = x_positions + (method_index - 0.5) * bar_width                                     ### PLOTTING
    bars = axis.bar(offsets, counts, width=bar_width, label=method)                               ### PLOTTING
    axis.bar_label(bars, labels=[f"{int(count)}/50" for count in counts], padding=3)             ### PLOTTING
axis.set(                                                                                         ### PLOTTING
    title="Unpenalized elbow-fit comparison",                                                   ### PLOTTING
    xlabel=r"true AR order $p$",                                                               ### PLOTTING
    ylabel="correct selections out of 50",                                                    ### PLOTTING
    xticks=x_positions, xticklabels=["AR(1)", "AR(2)", "AR(3)"], ylim=(0, 55),              ### PLOTTING
)                                                                                                 ### PLOTTING
axis.legend()                                                                                      ### PLOTTING
elbow_figure_path = (                                                                              ### PLOTTING
    ORDER_SELECTION_IMAGES_DIR / "cross_process"                                                ### PLOTTING
    / "unpenalized_elbow_fits_p1_p2_p3_50_each_T2000.png"                                        ### PLOTTING
)                                                                                                 ### PLOTTING
fig.savefig(elbow_figure_path, dpi=200, bbox_inches="tight")                                     ### PLOTTING
plt.show()                                                                                         ### PLOTTING

print("Correct selections out of 50")                                                          ### REPORTING
print(elbow_accuracy.to_string())                                                                  ### REPORTING


**Interpretation.** This block is deliberately limited to the two unpenalized curve-fitting rules. The classical construction has a structural lower bound of $\widehat p\ge2$ because both disjoint affine segments need at least two observations. The modified affine-plus-constant construction removes that guardrail by sharing the candidate plateau-onset point between its two fits. Numerical results should be interpreted as a comparison of these two model shapes, without tuning an order penalty.


## One complete AR(2) minimum-width pipeline

This reproducible example generates one stationary AR(2) process with innovations $\varepsilon_t\sim\operatorname{Unif}[-1,1]$, computes $\widehat m_k$ for $k=1,\ldots,10$ using three-start projected subgradient descent, and applies both unpenalized elbow fits. The population minimum is displayed for $k\ge p+1=3$, where $m_k=m^*=2/\lVert(1,-\phi_1,-\phi_2)\rVert_2$. Two cross-evaluated curves distinguish finite-sample width error from directional optimization error: $\widehat w_k(u_k^{\min})$ evaluates the empirical cloud in a population-minimizing direction, whereas $w_k(\widehat u_k)$ evaluates population width in the empirically minimizing direction. The $k=2$ population minimizer is obtained by one-dimensional numerical minimization.


In [ ]:
single_ar2_seed = 20260814
single_ar2_replication = 1
single_ar2_T = 2_000
single_ar2_burnin = 1_000
single_ar2_k_values = np.arange(1, 11)
single_ar2_generator = np.random.default_rng(np.random.SeedSequence([
    single_ar2_seed, single_ar2_replication, 0
]))
single_ar2_phi = jones_ar_coefficients(
    2, rng=single_ar2_generator
)
single_ar2_series = simulate_uniform_ar(
    single_ar2_phi, single_ar2_T, single_ar2_burnin, 1.0,
    rng=single_ar2_generator,
)
_, single_ar2_widths, _, single_ar2_empirical_directions = (
    batched_geometric_width_curves(
        single_ar2_series,
        single_ar2_k_values,
        seed=single_ar2_seed,
        replication=single_ar2_replication,
        starts=3,
        iterations=500,
        initial_step=0.2,
        geometric_decay=0.99,
        return_directions=True,
    )
)
single_ar2_selections = {
    "Two affine lines": two_affine_l_method_selector(
        single_ar2_k_values, single_ar2_widths
    ),
    "Affine + constant": affine_constant_l_method_selector(
        single_ar2_k_values, single_ar2_widths
    ),
}
single_ar2_population_width = 2.0 / np.linalg.norm(
    np.r_[1.0, -single_ar2_phi]
)
single_ar2_population_directions = []
single_ar2_empirical_at_population_min = []
single_ar2_population_at_empirical_min = []
for k, empirical_direction in zip(
    single_ar2_k_values, single_ar2_empirical_directions
):
    population_direction = population_minimizing_ar2_direction(
        single_ar2_phi, int(k)
    )
    cloud_k = sliding_window_point_cloud(single_ar2_series, int(k))
    single_ar2_population_directions.append(population_direction)
    single_ar2_empirical_at_population_min.append(
        np.ptp(cloud_k @ population_direction)
    )
    single_ar2_population_at_empirical_min.append(
        theoretical_ar_width_in_direction(
            single_ar2_phi, empirical_direction, half_width=1.0
        )
    )
single_ar2_empirical_at_population_min = np.asarray(
    single_ar2_empirical_at_population_min
)
single_ar2_population_at_empirical_min = np.asarray(
    single_ar2_population_at_empirical_min
)

single_ar2_pipeline_results = pd.DataFrame({                                                        ### REPORTING
    "k": single_ar2_k_values,                                                                       ### REPORTING
    "empirical_width": single_ar2_widths,                                                           ### REPORTING
    "population_width": np.where(                                                                   ### REPORTING
        single_ar2_k_values >= 3, single_ar2_population_width, np.nan                               ### REPORTING
    ),                                                                                              ### REPORTING
    "empirical_at_population_min": (                                                                ### REPORTING
        single_ar2_empirical_at_population_min                                                      ### REPORTING
    ),                                                                                              ### REPORTING
    "population_at_empirical_min": (                                                                ### REPORTING
        single_ar2_population_at_empirical_min                                                      ### REPORTING
    ),                                                                                              ### REPORTING
})                                                                                                  ### REPORTING

fig, axis = plt.subplots(figsize=(10.5, 6.4), constrained_layout=True)                              ### PLOTTING
axis.plot(                                                                                          ### PLOTTING
    single_ar2_k_values, single_ar2_widths, color="#1764ab",                                        ### PLOTTING
    marker="o", markersize=7.5, linewidth=2.3, zorder=4,                                            ### PLOTTING
    label=r"empirical $\widehat m_k$",                                                              ### PLOTTING
)                                                                                                   ### PLOTTING
axis.plot(                                                                                          ### PLOTTING
    single_ar2_k_values[2:],                                                                        ### PLOTTING
    np.full(len(single_ar2_k_values) - 2, single_ar2_population_width),                             ### PLOTTING
    color="black", linestyle="--", linewidth=2.5, zorder=2,                                         ### PLOTTING
    label=r"population $m_k=m^*$ for $k\geq3$",                                                     ### PLOTTING
)                                                                                                   ### PLOTTING
axis.plot(                                                                                          ### PLOTTING
    single_ar2_k_values, single_ar2_population_at_empirical_min,                                    ### PLOTTING
    color="#e66101", marker="s", markersize=5.8,                                                    ### PLOTTING
    linestyle="--", linewidth=2.0, zorder=3,                                                        ### PLOTTING
    label=r"population at empirical minimizer $w_k(\widehat u_k)$",                                 ### PLOTTING
)                                                                                                   ### PLOTTING
selector_styles = {                                                                                 ### PLOTTING
    "Two affine lines": ("#c51b2b", "-."),                                                     ### PLOTTING
    "Affine + constant": ("#6a3d9a", ":"),                                                      ### PLOTTING
}                                                                                                   ### PLOTTING
for method, selection in single_ar2_selections.items():                                             ### PLOTTING
    color, linestyle = selector_styles[method]                                                      ### PLOTTING
    axis.axvline(                                                                                    ### PLOTTING
        selection["selected_q"], color=color, linestyle=linestyle,                                 ### PLOTTING
        linewidth=1.8, zorder=1,                                                                    ### PLOTTING
        label=(                                                                                     ### PLOTTING
            f"{method}: q={selection['selected_q']} "                                              ### PLOTTING
            f"(p={selection['selected_order']})"                                                  ### PLOTTING
        ),                                                                                          ### PLOTTING
    )                                                                                               ### PLOTTING
axis.text(                                                                                          ### PLOTTING
    0.975, 0.965,                                                                                   ### PLOTTING
    rf"$\phi_1={single_ar2_phi[0]:.3f}$, "                                                          ### PLOTTING
    + rf"$\phi_2={single_ar2_phi[1]:.3f}$" + "\n"                                                   ### PLOTTING
    + rf"$T={single_ar2_T:,}$, $K={single_ar2_k_values[-1]}$",                                      ### PLOTTING
    transform=axis.transAxes, ha="right", va="top", fontsize=11,                                    ### PLOTTING
    bbox={"boxstyle": "round,pad=0.45", "facecolor": "white",                                       ### PLOTTING
          "edgecolor": "0.75", "alpha": 0.94},                                                      ### PLOTTING
)                                                                                                   ### PLOTTING
axis.set(                                                                                           ### PLOTTING
    title="One AR(2) process: population and empirical widths with selected elbow",                 ### PLOTTING
    xlabel=r"embedding dimension $k$",                                                              ### PLOTTING
    ylabel=r"directional width",                                                                    ### PLOTTING
    xticks=single_ar2_k_values,                                                                     ### PLOTTING
)                                                                                                   ### PLOTTING
axis.grid(axis="y", color="0.88", linewidth=0.8)                                                    ### PLOTTING
axis.spines[["top", "right"]].set_visible(False)                                                    ### PLOTTING
axis.legend(loc="upper right", bbox_to_anchor=(0.995, 0.82), fontsize=9)                              ### PLOTTING
single_ar2_pipeline_figure_path = (                                                                 ### PLOTTING
    ORDER_SELECTION_IMAGES_DIR / "ar2" / "ar2_pipeline_elbow_example_T2000.png"                     ### PLOTTING
)                                                                                                   ### PLOTTING
fig.savefig(                                                                                        ### PLOTTING
    single_ar2_pipeline_figure_path, dpi=220, bbox_inches="tight"                                   ### PLOTTING
)                                                                                                   ### PLOTTING
fig.savefig(                                                                                        ### PLOTTING
    EXPERIMENT_OVERVIEWS_DIR / "AR2_experiment.png",                                                ### PLOTTING
    dpi=220, bbox_inches="tight",                                                                   ### PLOTTING
)                                                                                                   ### PLOTTING
plt.show()                                                                                          ### PLOTTING

print(f"AR coefficients = {single_ar2_phi}")                                                        ### REPORTING
print(f"Population plateau m* = {single_ar2_population_width:.6f}")                                 ### REPORTING
for method, selection in single_ar2_selections.items():                                             ### REPORTING
    print(                                                                                           ### REPORTING
        f"{method}: selected q = {selection['selected_q']}; "                                       ### REPORTING
        f"selected AR order p = {selection['selected_order']}; "                                   ### REPORTING
        f"fit score = {selection['score']:.6f}"                                                    ### REPORTING
    )                                                                                                ### REPORTING
print(single_ar2_pipeline_results.to_string(index=False))                                           ### REPORTING


**Interpretation.** The two vertical markers report the elbows selected from the same empirical minimum-width curve, using only the two retained unpenalized fits. At and beyond the true kink $k=3$, $\widehat w_k(u_k^{\min})$ is almost indistinguishable from the population plateau $m^*$, whereas $w_k(\widehat u_k)$ becomes substantially larger at higher dimensions. This divergence shows that the empirical minimization increasingly exploits finite-sample directional gaps as the embedding dimension grows.

![One AR(2) minimum-width pipeline with population and empirical widths](outputs/images/order_selection/ar2/ar2_pipeline_elbow_example_T2000.png)


## Candidate fit scores for the displayed AR(2) curve

The final table exposes the unpenalized fit score at every admissible candidate. It is intended to make the selection step auditable; no additional selector or benchmark is applied in this notebook.


In [ ]:
single_ar2_candidate_scores = pd.concat(
    {
        method: selection["candidate_scores"]
        for method, selection in single_ar2_selections.items()
    },
    names=["method", "candidate"],
).reset_index(level="method").reset_index(drop=True)
display(single_ar2_candidate_scores)                                                               ### REPORTING


Only the two unpenalized fits above are part of the present elbow-selection analysis. No additional selection rule or benchmark is included at this stage.
